# Play Procedural Frozen Lake

This notebook lets you play `Procedural-FrozenLake-v1` yourself. Each lake is generated from the procedural map generator, then displayed after every move so you can see where you are, where the holes are, and when an episode ends.

Use the D-pad buttons to move. If widgets are unavailable, call `play("left")`, `play("down")`, `play("right")`, or `play("up")` in a code cell.

In [28]:
import io
import os
import random

# FrozenLake renders through pygame. These defaults let rgb_array rendering work
# in headless notebook servers as well as desktop sessions.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")

import numpy as np
import torch
from IPython.display import clear_output, display
from PIL import Image as PILImage

from mouse_envs import EnvConfig, make_env

## Build a Lake

The config below asks the custom environment for a generated, valid map and `rgb_array` rendering. The rendered frame is the display: it shows the lake, the player, holes, and goals after each move.

The env follows mouse-env's reset-free protocol: the first `step()` returns a reset frame, and the first click after a terminal state returns the next episode's reset frame.

In [29]:
ACTION_TO_ID = {
    "left": 0,
    "down": 1,
    "right": 2,
    "up": 3,
}
ID_TO_ACTION = {v: k for k, v in ACTION_TO_ID.items()}
ARROWS = {
    "left": "<-",
    "down": "v",
    "right": "->",
    "up": "^",
}


def make_play_env(*, map_seed=None):
    cfg = EnvConfig(
        id="Procedural-FrozenLake-v1",
        reset_seed=0,
        episodes_per_task=0,
        q_star_source={"provider": "env_q_star"},
        kwargs={
            "render_mode": "rgb_array",
            "map_seed": map_seed,
            "min_width": 5,
            "max_width": 7,
            "min_height": 5,
            "max_height": 7,
            "hole_prob": 0.50,
            "min_hops": 5,
            "is_slippery": False,
            "max_episode_steps": 100,
        },
    )
    return make_env(cfg)

In [30]:
env = None
outputs = None
map_seed = None


def close_current_env():
    global env
    if env is not None:
        env.close()
        env = None


def current_output():
    if outputs is None:
        raise RuntimeError("Call new_lake() before playing.")
    return outputs[0]


def scalar_item(value):
    if hasattr(value, "item"):
        return value.item()
    return np.asarray(value).reshape(-1)[0]


def current_state(output):
    return int(scalar_item(output["observation"]))


def action_input(action_name):
    return [{"action": torch.tensor(ACTION_TO_ID[action_name], dtype=torch.int64)}]


def expert_hint(output):
    q_star = output.get("info_q_star")
    if q_star is None:
        return ""
    best_action = ID_TO_ACTION[int(np.argmax(q_star))]
    return f" Expert hint: {best_action} ({ARROWS[best_action]})."


def status_text(output, *, last_action=None):
    done_code = int(scalar_item(output["done"]))
    reward = float(scalar_item(output["reward"]))
    time = int(scalar_item(output["time"]))
    state = current_state(output)

    prefix = f"seed={map_seed} | state={state} | time={time} | reward={reward:.2f}"
    if last_action is not None:
        prefix += f" | last action={last_action} {ARROWS[last_action]}"

    if done_code == 0 and time == 0:
        return prefix + " | reset frame: press a direction to move." + expert_hint(output)
    if done_code == 0:
        return prefix + " | keep going." + expert_hint(output)
    if done_code in (1, 3):
        return prefix + " | episode terminated. Press any direction once to reset."
    return prefix + " | episode truncated. Press any direction once to reset."


def show(output=None, *, last_action=None):
    if output is None:
        output = current_output()

    clear_output(wait=True)
    frames = env.render() if env is not None else []
    if frames:
        display(PILImage.fromarray(frames[0]))

    print(status_text(output, last_action=last_action))

In [31]:
def new_lake(seed=None, *, display_now=True):
    """Generate a new procedural lake and show the initial reset frame."""
    global env, outputs, map_seed

    close_current_env()
    map_seed = random.randrange(1_000_000) if seed is None else int(seed)
    env = make_play_env(map_seed=map_seed)
    outputs = env.step(env.sample_random_inputs())

    if display_now:
        show(outputs[0])
    return outputs[0]


def play(action_name, *, display_now=True):
    """Move the player one step. Use left, down, right, or up."""
    global outputs

    action_name = str(action_name).lower().strip()
    if action_name not in ACTION_TO_ID:
        valid = ", ".join(ACTION_TO_ID)
        raise ValueError(f"Unknown action {action_name!r}; use one of: {valid}")

    outputs = env.step(action_input(action_name))
    if display_now:
        show(outputs[0], last_action=action_name)
    return outputs[0]


def play_left():
    return play("left")


def play_down():
    return play("down")


def play_right():
    return play("right")


def play_up():
    return play("up")

## Play

Run the next cell to create the lake. Use the D-pad buttons to move, or call the same `play(...)` function from code.

In [32]:
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None


if widgets is None:
    print("ipywidgets is not installed, so use function calls to play:")
    print('  play("left"), play("down"), play("right"), play("up")')
    print("Generate a fresh map with new_lake().")
    new_lake()
else:
    def frame_png_bytes():
        frames = env.render() if env is not None else []
        if not frames:
            return b""
        image = PILImage.fromarray(frames[0])
        buffer = io.BytesIO()
        image.save(buffer, format="PNG")
        return buffer.getvalue()

    def redraw(output=None, *, last_action=None):
        if output is None:
            output = current_output()
        game_image.value = frame_png_bytes()
        status_label.value = status_text(output, last_action=last_action)

    def move(action_name):
        play(action_name, display_now=False)
        redraw(current_output(), last_action=action_name)

    def on_move(action_name):
        def callback(_):
            move(action_name)
        return callback

    def on_new_lake(_=None):
        new_lake(display_now=False)
        redraw(current_output())

    button_layout = widgets.Layout(width="80px")
    spacer_layout = widgets.Layout(width="80px")
    up_button = widgets.Button(description="Up", layout=button_layout)
    left_button = widgets.Button(description="Left", layout=button_layout)
    down_button = widgets.Button(description="Down", layout=button_layout)
    right_button = widgets.Button(description="Right", layout=button_layout)
    new_button = widgets.Button(description="New lake", button_style="info")
    game_image = widgets.Image(
        format="png",
        layout=widgets.Layout(width="420px"),
    )
    status_label = widgets.Label(layout=widgets.Layout(width="680px"))

    up_button.on_click(on_move("up"))
    left_button.on_click(on_move("left"))
    down_button.on_click(on_move("down"))
    right_button.on_click(on_move("right"))
    new_button.on_click(on_new_lake)

    dpad = widgets.VBox([
        widgets.HBox([
            widgets.Label("", layout=spacer_layout),
            up_button,
            widgets.Label("", layout=spacer_layout),
        ]),
        widgets.HBox([left_button, down_button, right_button]),
    ])
    game_view = widgets.VBox([game_image, status_label])
    controls = widgets.VBox([dpad, new_button])
    display(widgets.VBox([controls, game_view]))

    new_lake(display_now=False)
    redraw(current_output())